数据来源：https://www.kaggle.com/competitions/ml2021spring-hw3/data

库配置

In [1]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np

# 库配置
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
os.environ["TORCH_HOME"] = "./pretrained_models" #改变预训练模型下载路径

数据处理

In [2]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop((128, 128)),# 截取原图像的一部分然后将截取下来的这个区域强制缩放到 128 x 128 像素的大小
    transforms.RandomChoice(# 以下选项中随机选择一项
        [transforms.AutoAugment(),# 默认的自动增强策略
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),# 专门针对 CIFAR-10(小尺寸自然图像)找出的最优增强策略
        transforms.AutoAugment(transforms.AutoAugmentPolicy.SVHN)] # 专门针对 SVHN（街景门牌号数字）找出的最优增强策略
    ),
    transforms.RandomHorizontalFlip(p=0.5),# 随机水平翻转
    transforms.ColorJitter(brightness=0.5),# 随机改变图像的亮度。0.5 意味着亮度会在原始亮度的 0.5 倍（变暗 50%）到 1.5 倍（变亮 50%）之间随机浮动
    transforms.RandomRotation(5),# 在 -5 度到 +5 度之间随机旋转图片
    transforms.ToTensor(),# 转换为张量
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),# 将图像调整为 128 x 128 像素的大小
    transforms.ToTensor(),# 转换为张量
])

数据加载

In [3]:
# ImageFolder 每次遍取数据时，默认都会返回一个元组：(数据, 标签)，返回的标签是一个整数，一一对应文件夹下的类别
train_set = datasets.ImageFolder(root="food-11/training/labeled", transform=train_transform)
validation_set = datasets.ImageFolder(root="food-11/validation", transform=test_transform)
unlabel_set = datasets.ImageFolder(root="food-11/training/unlabeled", transform=test_transform)
test_set = datasets.ImageFolder(root="food-11/testing", transform=test_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

手动实现卷积神经网络网络

In [4]:
class CnnClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2,padding=0),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2,padding=0),

            nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4,stride=4,padding=0),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  #展平
        x = self.fc_layers(x)
        return x

先只用有标签数据训练模型

In [5]:
epochs = 50
best_validation_accuracy = 0
# model = CnnClassifier(num_classes=11).to(device) # 手动实现的效果并不好，调用预训练的模型
model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 11)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(epochs):
    model.train()
    epoch_total_loss = 0
    for images, labels in train_loader:
        images,labels = images.to(device),labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        epoch_total_loss += loss.item()

    # 每一轮训练完后评估
    model.eval()
    correct = 0
    accuracy = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images,labels = images.to(device),labels.to(device)
            output = model(images)
            predict = output.argmax(dim=1, keepdim=True)
            correct += predict.eq(labels.view_as(predict)).sum().item()
        accuracy = correct / len(validation_set)
        print(f"Epoch: {epoch+1}/{epochs}, epoch_total_loss: {epoch_total_loss}, accuracy：{accuracy}")

        if accuracy > best_validation_accuracy:
            best_validation_accuracy = accuracy
            torch.save(model.state_dict(), './best_model.pth')
            print(f"Best model saved at epoch {epoch + 1}, accuracy: {accuracy}",end="\n\n")


Epoch: 1/50, epoch_total_loss: 77.87975144386292, accuracy：0.5909090909090909
Best model saved at epoch 1, accuracy: 0.5909090909090909

Epoch: 2/50, epoch_total_loss: 62.22072923183441, accuracy：0.6803030303030303
Best model saved at epoch 2, accuracy: 0.6803030303030303

Epoch: 3/50, epoch_total_loss: 54.502160251140594, accuracy：0.6924242424242424
Best model saved at epoch 3, accuracy: 0.6924242424242424

Epoch: 4/50, epoch_total_loss: 52.02124971151352, accuracy：0.6439393939393939
Epoch: 5/50, epoch_total_loss: 50.109272956848145, accuracy：0.7378787878787879
Best model saved at epoch 5, accuracy: 0.7378787878787879

Epoch: 6/50, epoch_total_loss: 49.33179718255997, accuracy：0.6833333333333333
Epoch: 7/50, epoch_total_loss: 48.88484483957291, accuracy：0.7196969696969697
Epoch: 8/50, epoch_total_loss: 46.315080881118774, accuracy：0.7136363636363636
Epoch: 9/50, epoch_total_loss: 46.415147960186005, accuracy：0.746969696969697
Best model saved at epoch 9, accuracy: 0.746969696969697

E

加载用有标签数据训练的模型为无标签数据加上伪标签

In [6]:
unlabel_loader = DataLoader(unlabel_set, batch_size=64, shuffle=False)
model.load_state_dict(torch.load('./best_model.pth'))
model.eval()
pseudo_dataset = [] # 用来存放 (图片, 伪标签) 的列表
threshold = 0.95    # 只信任概率大于 95% 的预测

with torch.no_grad():
    for images, _ in unlabel_loader:
        images = images.to(device)
        outputs = model(images)

        #用 Softmax 将输出转化为 0~1 的概率
        probs = torch.softmax(outputs, dim=-1)

        #拿到这批数据的最大概率和对应的预测分类
        max_probs, predicted_labels = torch.max(probs, dim=-1)

        #逐个筛选，只把置信度大于95%的图片加入我们的伪标签数据集中
        for i in range(len(probs)):
            if max_probs[i] > threshold:
                img_cpu = images[i].cpu()# 转移回内存，否则显存可能不够
                label_cpu = predicted_labels[i].cpu().item()

                # 打包成元组存入列表，完美契合 PyTorch 数据集格式
                pseudo_dataset.append((img_cpu, label_cpu))

print(f"从无标签数据中抽取了{len(pseudo_dataset)} 张置信度大于95%的伪标签数据！")

从无标签数据中抽取了3523 张置信度大于95%的伪标签数据！


合并数据

In [7]:
mixed_dataset = ConcatDataset([train_set, pseudo_dataset])

#重新打包成 DataLoader
mixed_loader = DataLoader(mixed_dataset, batch_size=64, shuffle=True)

print(f"合并后有 {len(mixed_dataset)} 张图片用来训练")

合并后有 6603 张图片用来训练了！


使用合并后的数据进行训练

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)# 在原有模型上进一步精细训练，所以调小学习率
for epoch in range(epochs):
    model.train()
    epoch_total_loss = 0
    for images, labels in mixed_loader:
        images,labels = images.to(device),labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        epoch_total_loss += loss.item()

    # 每一轮训练完后评估
    model.eval()
    correct = 0
    accuracy = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images,labels = images.to(device),labels.to(device)
            output = model(images)
            predict = output.argmax(dim=1, keepdim=True)
            correct += predict.eq(labels.view_as(predict)).sum().item()
        accuracy = correct / len(validation_set)
        print(f"Epoch: {epoch+1}/{epochs}, epoch_total_loss: {epoch_total_loss}, accuracy：{accuracy}")

        if accuracy > best_validation_accuracy:
            best_validation_accuracy = accuracy
            torch.save(model.state_dict(), './best_mixed_model.pth')
            print(f"Best model saved at epoch {epoch + 1}, accuracy: {accuracy}",end="\n\n")


Epoch: 1/50, epoch_total_loss: 24.81363881379366, accuracy：0.8272727272727273
Best model saved at epoch 1, accuracy: 0.8272727272727273

Epoch: 2/50, epoch_total_loss: 21.87543110176921, accuracy：0.8424242424242424
Best model saved at epoch 2, accuracy: 0.8424242424242424

Epoch: 3/50, epoch_total_loss: 19.935286471620202, accuracy：0.843939393939394
Best model saved at epoch 3, accuracy: 0.843939393939394

Epoch: 4/50, epoch_total_loss: 20.486708629876375, accuracy：0.8393939393939394
Epoch: 5/50, epoch_total_loss: 18.15581458620727, accuracy：0.85
Best model saved at epoch 5, accuracy: 0.85

Epoch: 6/50, epoch_total_loss: 18.57147600222379, accuracy：0.85
Epoch: 7/50, epoch_total_loss: 16.99943098425865, accuracy：0.8378787878787879
Epoch: 8/50, epoch_total_loss: 17.796917164698243, accuracy：0.853030303030303
Best model saved at epoch 8, accuracy: 0.853030303030303

Epoch: 9/50, epoch_total_loss: 16.795745556242764, accuracy：0.8575757575757575
Best model saved at epoch 9, accuracy: 0.8575

合并数据训练后的模型在验证集上的表现更好，使用这个模型在测试集上进行预测并保存结果

In [9]:
model.load_state_dict(torch.load('./best_mixed_model.pth'))
model.eval()
predictions = []
with torch.no_grad():
    for images, _ in test_loader:
        images= images.to(device)
        outputs = model(images)
        labels = outputs.argmax(dim=1).cpu().numpy()
        predictions.append(labels)

import pandas as pd
predictions = np.concatenate(predictions) # 不同批次的预测结果首尾拼接
# 获取对应的文件名作为 Id
file_names = [os.path.basename(path).split('.')[0] for path, _ in test_set.samples]
df = pd.DataFrame({'Id': file_names, 'Category': predictions})
df.to_csv('prediction.csv', index=False)
print("预测结果已保存至 prediction.csv")

预测结果已保存至 prediction.csv
